<a href="https://colab.research.google.com/github/miso-20/ESSA/blob/main/ESAA_YB_WEEK_12_2-review3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **수상작 리뷰**

2021 농산물 가격예측 AI 경진대회

https://dacon.io/competitions/official/235801/codeshare/4063?page=1&dtype=recent

## **주제**
LSTM모델과 시계열 분해 기법을 활용한 농산물 가격 예측

## **데이터**

1. public_data : public leaderboard용 데이터

    1-1. train.csv : 베이스라인 코드용으로 가공된 학습 데이터

    - date: 일자
    - 요일: 요일
    - 품목_거래량(kg): 해당 품목의 거래량
    - 품목_가격(원/kg): 해당 품목의 kg당 가격
    - 품목_가격 산출 방식 : 품목 또는 품종의 총 거래금액/총 거래량 (※취소된 거래내역 제외)

    1-2. test_files : 베이스라인 코드용으로 가공된 테스트 데이터(추론일자별 분리, ex.test_2020-08-29.csv => 2020년 8월 29일 추론에 사용 가능 데이터)

    1-3. train_AT_TSALET_ALL : 학습용 전국 도매시장 거래정보 데이터(train.csv 생성에 사용)

    - SALEDATE: 경락 일자
    - WHSAL_NM: 도매시장
    - CMP_NM: 법인
    - PUM_NM: 품목
    - KIND_NM: 품종
    - DAN_NM: 단위
    - POJ_NM: 포장
    - SIZE_NM: 크기
    - LV_NM: 등급
    - SAN_NM: 산지
    - DANQ: 단위중량
    - QTY: 물량
    - COST: 단가
    - TOT_QTY: 총물량 (음수로 집계된 값은 거래 취소 내역)
    - TOT_AMT: 총금액

    1-4. test_AT_TSALET_ALL : 추론용 전국 도매시장 거래정보 데이터(test_files 생성에 사용)



2. private_data : private leaderboard용 데이터, public leaderboard 학습 및 추론 사용 불가

	2-1. private.csv : 베이스라인 코드용으로 가공된 데이터

	2-2. AT_TSALET_ALL : 2021년 8월까지의 전국 도매시장 거래정보 데이터(private.csv 생성에 사용)

## **코드 흐름**


### 1. 데이터 수집 및 전처리

- 기존 CSV 파일과 API 호출 데이터를 하나의 DataFrame(df1, df2)으로 병합

- 0값을 결측치(NaN)로 변환 후, interpolate() 메서드를 사용해 선형 보간하여 빈 값을 채움

### 2. 피처 엔지니어링 및 타겟 설정

- 요일 변수는 원핫 인코딩(get_dummies)을 적용하여 수치형으로 변환

- 예측 목표인 1주, 2주, 4주 후의 가격은 shift() 함수를 사용하여 타겟 변수로 생성

- 시계열 분해(STL)를 통해 추출한 잔차(resid)를 독립적인 피처로 추가


### 3. LSTM 기반 모델 학습

- MinMaxScaler로 피처 스케일링 진행

- GPU 기반의 CuDNNLSTM 레이어를 활용하여 시계열 특성을 반영한 예측 모델 학습(조기 종료 적용)














**주요 코드**

In [ ]:
# 모델 타겟 변수 shift 처리 및 STL 시계열 분해 잔차 추출
def set_target(self,week):
    if week == 1:
        self.df['target'] = self.df[self.name1].shift(-8)
    # ... (2주, 4주 shift 처리 생략) ...
    self.df['resid'] = 0
    stl = STL(self.df[['date', self.name1]].set_index('date'), period=12)
    res = stl.fit()
    self.df['resid'] = res.resid.values

## **새롭게 알게 된 내용 / 어려운 내용 / 배울 점**

- 단순히 과거의 가격과 거래량만 학습에 사용하는 것을 넘어 statsmodels의 STL 모듈을 사용해 데이터의 계절성(Seasonality)과 추세(Trend)를 분리하고, 불규칙한 변동을 의미하는 잔차(Residual) 값을 모델의 핵심 피처로 활용하는 접근법이 인상적이었다.

- 농산물 거래 특성상 휴장일 등으로 인해 발생하는 0값을 단순히 지우거나 평균값으로 채우지 않고, replace(0, np.NaN) 후 interpolate()를 사용하여 앞뒤 시계열 흐름에 맞게 선형 보간하는 전처리 방식을 새롭게 배웠다.

- 21개의 각기 다른 품목에 대해 1주, 2주, 4주(총 3가지)의 타겟 기간을 설정하고, 이를 이중 For 루프를 통해 객체(Nong1)를 초기화하며 독립적인 딥러닝 모델을 수십 번 학습시키는 구조는 메모리와 실행 시간 관리가 까다롭고 전체적인 파이프라인 설계 난이도가 높게 느껴졌다.
